# Member Churn Prediction Model

Trains a binary classification model to predict member churn using features from `ML_FEATURES.ML_MEMBER_CHURN_FEATURES`.

**Target:** `is_churned` (1 = membership expired/lapsed/grace period)

**Approach:** XGBoost classifier registered to Snowflake Model Registry

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np

session = get_active_session()
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

In [ ]:
%%sql -r churn_data
SELECT *
FROM NS11MM_DW_DEV.ML_FEATURES.ML_MEMBER_CHURN_FEATURES
WHERE contact_id IS NOT NULL

In [ ]:
df = churn_data
print(f"Dataset shape: {df.shape}")
print(f"\nChurn distribution:")
print(df['IS_CHURNED'].value_counts(normalize=True))
print(f"\nFeature columns:")
print(df.columns.tolist())

In [ ]:
feature_cols = [
    'DAYS_SINCE_LAST_INTERACTION',
    'DAYS_SINCE_LAST_VISIT',
    'DONATION_TOTAL_YTD',
    'TOTAL_TICKET_SPEND',
    'TOTAL_RETAIL_SPEND',
    'TOTAL_LIFETIME_VALUE',
    'EMAIL_OPENS',
    'EMAIL_CLICKS',
    'EMAIL_CLICK_THROUGH_RATE',
]

categorical_cols = ['MEMBERSHIP_TYPE', 'DONOR_TIER', 'ENGAGEMENT_SEGMENT']
target_col = 'IS_CHURNED'

df_encoded = pd.get_dummies(df[feature_cols + categorical_cols + [target_col]], columns=categorical_cols, drop_first=True)

X = df_encoded.drop(columns=[target_col]).fillna(0)
y = df_encoded[target_col]

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"Churn rate: {y.mean():.2%}")

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))
print(f"\nROC AUC: {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feature_importance['feature'], feature_importance['importance'])
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 Features for Member Churn Prediction')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml._internal.utils.identifier import get_inferred_name

reg = Registry(session=session, database_name='NS11MM_DW_DEV', schema_name='ML_REGISTRY')

metrics = {
    'roc_auc': float(roc_auc_score(y_test, y_prob)),
    'test_size': len(y_test),
    'churn_rate': float(y.mean()),
    'n_features': X_train.shape[1]
}

mv = reg.log_model(
    model=model,
    model_name='member_churn_xgboost',
    version_name='v1',
    sample_input_data=X_test.head(10),
    metrics=metrics,
    comment='XGBoost member churn classifier trained on ML_MEMBER_CHURN_FEATURES'
)

print(f"Model registered: {mv.model_name} v{mv.version_name}")
print(f"Metrics: {metrics}")